# 01 - Deploy Infrastructure

Deploy `infra/main.bicep` and write outputs to `env/.env` for later notebooks.

**Estimated time:** 45-70 minutes

> **Important:** VPN gateway provisioning alone is typically **30-45 minutes**.


## Step 1 - Set deployment variables


In [ ]:
RESOURCE_GROUP = "rg-search01-private-endpoints"
LOCATION = "eastus2"
DEMO_ID = "search01-private-endpoints"

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Demo ID        : {DEMO_ID}")


## Step 2 - Create resource group


In [ ]:
!az group create --name {RESOURCE_GROUP} --location {LOCATION} --output table


## Step 3 - Deploy Bicep


In [ ]:
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found.")

deployer_principal_id = subprocess.run([az_cmd, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"], capture_output=True, text=True, check=True).stdout.strip()
tenant_id = subprocess.run([az_cmd, "account", "show", "--query", "tenantId", "-o", "tsv"], capture_output=True, text=True, check=True).stdout.strip()

cmd = [
    az_cmd, "deployment", "group", "create",
    "--resource-group", RESOURCE_GROUP,
    "--template-file", "../infra/main.bicep",
    "--parameters",
    f"location={LOCATION}",
    f"demoId={DEMO_ID}",
    f"deployerPrincipalId={deployer_principal_id}",
    f"tenantId={tenant_id}",
    "--output", "json"
]
result = subprocess.run(cmd, capture_output=True, text=True, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Deployment failed.\nSTDERR:\n{result.stderr}\nSTDOUT:\n{result.stdout}")
print("Deployment succeeded")


## Step 4 - Read deployment outputs


In [ ]:
import json
import subprocess

outputs_payload = subprocess.run([az_cmd, "deployment", "group", "list", "--resource-group", RESOURCE_GROUP, "--query", "[0].properties.outputs", "-o", "json"], capture_output=True, text=True, check=True).stdout
outputs = json.loads(outputs_payload or "{}")
required = ["storageAccountName", "containerName", "storageAccountResourceId", "searchServiceName", "searchEndpoint", "searchResourceId", "vnetGatewayName", "dnsForwarderPrivateIp", "vnetId", "snetPePrefix", "vpnClientAddressPool", "resourceGroupName", "tenantId"]
missing = [k for k in required if k not in outputs]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
resolved = {k: outputs[k]["value"] for k in required}
for k, v in resolved.items():
    print(f"{k}: {v}")


## Step 5 - Write `env/.env`


In [ ]:
from pathlib import Path

subscription_id = subprocess.run([az_cmd, "account", "show", "--query", "id", "-o", "tsv"], capture_output=True, text=True, check=True).stdout.strip()
lines = [
    "# Auto-generated by 01_deploy_infra.ipynb - do not commit",
    f"AZURE_SUBSCRIPTION_ID={subscription_id}",
    f"AZURE_TENANT_ID={resolved['tenantId']}",
    f"AZURE_RESOURCE_GROUP={resolved['resourceGroupName']}",
    f"AZURE_LOCATION={LOCATION}",
    f"SEARCH_DEMO_ID={DEMO_ID}",
    "",
    f"STORAGE_ACCOUNT_NAME={resolved['storageAccountName']}",
    f"STORAGE_CONTAINER_NAME={resolved['containerName']}",
    f"STORAGE_ACCOUNT_RESOURCE_ID={resolved['storageAccountResourceId']}",
    "",
    f"SEARCH_SERVICE_NAME={resolved['searchServiceName']}",
    f"SEARCH_ENDPOINT={resolved['searchEndpoint']}",
    f"SEARCH_RESOURCE_ID={resolved['searchResourceId']}",
    "",
    f"VNET_GATEWAY_NAME={resolved['vnetGatewayName']}",
    f"DNS_FORWARDER_PRIVATE_IP={resolved['dnsForwarderPrivateIp']}",
    f"VNET_ID={resolved['vnetId']}",
    f"SNET_PE_PREFIX={resolved['snetPePrefix']}",
    f"VPN_CLIENT_ADDRESS_POOL={resolved['vpnClientAddressPool']}",
]
env_path = Path("../env/.env")
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Written {env_path.resolve()}")


## Step 6 - Verify `env/.env`


In [ ]:
from pathlib import Path
env_path = Path("../env/.env")
if not env_path.exists():
    raise FileNotFoundError(env_path)
print(env_path.read_text(encoding="utf-8"))


---

Continue with `02_connect_vpn.ipynb`.
